In [2]:
from pathlib import Path
import json

import pandas as pd
from pysentimiento import create_analyzer
from transformers import MarianMTModel, MarianTokenizer


In [13]:
STEPS_PATH = Path("../www/data/steps.json")
OUTPUT_PATH = Path("../www/data/steps_enriched.json")

In [7]:
with STEPS_PATH.open(encoding="utf-8") as f:
    steps = json.load(f)

In [9]:
# Spanish sentiment analyzer (pysentimiento)
sentiment_analyzer = create_analyzer(task="sentiment", lang="es")

# Quick test
test = sentiment_analyzer.predict("Buenos días. Buenos días ¿Cómo están?")
test, test.probas


(AnalyzerOutput(output=POS, probas={POS: 0.522, NEU: 0.444, NEG: 0.034}),
 {'NEG': 0.034180738031864166,
  'NEU': 0.44428545236587524,
  'POS': 0.5215338468551636})

In [10]:
model_name = "Helsinki-NLP/opus-mt-es-en"
tokenizer = MarianTokenizer.from_pretrained(model_name)
mt_model = MarianMTModel.from_pretrained(model_name)

def translate_es_en(text: str) -> str:
    if not text:
        return ""
    batch = tokenizer([text], return_tensors="pt", truncation=True)
    generated = mt_model.generate(**batch, max_length=128)
    return tokenizer.decode(generated[0], skip_special_tokens=True)


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


tokenizer_config.json:   0%|          | 0.00/44.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/826k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/802k [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

/Users/jmcarias/Library/Python/3.9/lib/python/site-packages/transformers/models/marian/tokenization_marian.py:175: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


pytorch_model.bin:   0%|          | 0.00/312M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/312M [00:00<?, ?B/s]

In [12]:
def sentiment_score_es(text: str) -> float:
    """
    Returns a score in [-1, 1] using:
      score = P(pos) - P(neg)
    """
    if not text:
        return 0.0
    res = sentiment_analyzer.predict(text)
    pos = float(res.probas.get("POS", 0.0))
    neg = float(res.probas.get("NEG", 0.0))
    return pos - neg

for step in steps:
    quote = step.get("quote", "")
    
    # 1) Sentiment in [-1, 1]
    score = sentiment_score_es(quote)
    step["sentiment"] = round(score, 3)
    
    # 2) English translation of the quote
    step["quote_en"] = translate_es_en(quote)

# Quick preview
steps[0]

{'id': 1,
 'step': 1,
 'speaker': 'president',
 'silhouette': 'img/president.png',
 'sentiment': 0.487,
 'context': 'The President opens the morning conference with a calm greeting, setting a measured tone before any mention of the floods.',
 'quote': 'Buenos días. Buenos días ¿Cómo están?',
 'quote_en': 'Good morning, how are you?'}

In [14]:
# Save updated JSON
with OUTPUT_PATH.open("w", encoding="utf-8") as f:
    json.dump(steps, f, ensure_ascii=False, indent=2)

print(f"Saved enriched steps to: {OUTPUT_PATH.resolve()}")


Saved enriched steps to: /Users/jmcarias/Documents/CAPP30239/morning_pulse/www/data/steps_enriched.json
